In [ ]:
import MDAnalysis as mda
from MDAnalysis.analysis import *
import numpy as np
import pandas as pd
import plotly.express as px
import nglview as nv
import matplotlib.pyplot as plt
import math
from collections import Counter
from parmed import load_file


In [ ]:
water_rdf_df = pd.DataFrame()

for length in ['3_res','9_res']:
  
    for system in ['equal','charged','aromatic','polar','nonpolar']:
    
        if length == '3_res':
            name = '3res'
            if system == 'aromatic':
                u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/output.pdb',
                                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/traj.dcd',
                                f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/traj_2.dcd')
            else:
                traj = 'trajectory.dcd'
                u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/output.pdb',
                            f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/{traj}')
        else:
            name = '9res'
            traj = 'traj.dcd'
            u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/output.pdb',
                            f'/Volumes/potoyan-lab/will/gxg-project/gxg_comps/{length}/{system}/{traj}')

        a_group = u.select_atoms(f'resname HOH')
        b_group = u.select_atoms(f'resname HOH')

        rdf = mda.analysis.rdf.InterRDF(a_group, b_group, nbins = 100, range = [1,50], exclude_same="residue")
        rdf.run(step=1000)

        temp_df = pd.DataFrame()
        temp_df[f'{length}_{system}'] = rdf.rdf
        
        water_rdf_df = pd.concat([water_rdf_df, temp_df[f'{length}_{system}']], axis=1)



In [ ]:
temp = u.select_atoms('protein and name CA')

chain_length = Counter(temp.fragindices)[0]

middle = math.ceil(Counter(temp.fragindices)[0]/2)

n_res = u.select_atoms(f'resname ALA').n_residues

id_list = [middle]
for i in range(n_res)[1:]:
    id_list.append(middle+(chain_length*i))
    
selection = f'resid {middle} or resid '
for item in id_list:
    selection += str(item) + ' or resid '
selection = selection[0:-10]


In [ ]:
test = u.select_atoms(f'resname GLY and {selection}')

In [ ]:
Counter(test.residues.resnames)

In [ ]:
prot = u.select_atoms('protein')
view = nv.show_mdanalysis(test)
view.add_ball_and_stick()
view

In [ ]:
chain_length = 9
rep = 1

u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/chain_length/{chain_length}_res/rep_{rep}/output-nw.pdb',
                f'/Volumes/potoyan-lab/will/gxg-project/chain_length/{chain_length}_res/rep_{rep}/traj-nw.dcd')

aromatic = ['PHE', 'TYR', 'TRP']
polar = ['CYS','THR','GLN','ASN','SER']
nonpolar = ['VAL','ILE','ALA','LEU','PRO','GLY','MET']
positive = ['ARG','LYS','HIS']
negative = ['GLU','ASP']

# loop=0
# for chain_length in [9,15,21]:
for chain_length in [9]:
    u = mda.Universe(f'/Volumes/potoyan-lab/will/gxg-project/chain_length/{str(chain_length)}_res/rep_1/output-nw.pdb',
                     f'/Volumes/potoyan-lab/will/gxg-project/chain_length/{str(chain_length)}_res/rep_1/traj-nw.dcd')
    middle = math.ceil(chain_length/2)
    res_loop = 0
    for i in ['ALA','ARG','ASN','ASP','CYS','GLN','GLU','GLY','HIS','ILE',
              'LEU','LYS','MET','PHE','PRO','SER','THR','TRP','TYR','VAL']:
#         for i in nonpolar:
        a_group = u.select_atoms(f'resname {i} and name CA and resid {middle}')
        b_group = u.select_atoms(f'resname {i} and name CA and resid {middle}')
#         a_group = u.select_atoms(f'protein and name CA and resid {middle}')
#         b_group = u.select_atoms(f'protein and name CA and resid {middle}')
        rdf = mda.analysis.rdf.InterRDF(a_group, b_group, nbins = 100, range = [1,35], exclude_same="residue")
        rdf.run(step=100)
        if res_loop==0:
            temp_df = pd.DataFrame(index=rdf.bins)
            temp_df[f'{i}'] = rdf.rdf
        else:
             temp_df[f'{i}'] = rdf.rdf
#         values = temp_df.mean(axis=1).values
#         if loop==0:
#             rdf_df = pd.DataFrame(index=rdf.bins)
#             rdf_df[f'{chain_length}_{rep}'] = rdf.rdf
#         else:
#             rdf_df[f'{chain_length}_{rep}'] = rdf.rdf
        res_loop+=1


In [ ]:
temp_df

In [ ]:
px.line(temp_df)


In [ ]:
px.line(rdf_df)
